# Spin-1 XY numerical evidence: directed witness and deformation stability

This notebook is keyed to **Sec. III, Sec. IV, Sec. VI, and Appendix A.4** of the July 21, 2026 ICQMBS draft.  The analytical spin-1 XY section already derives the tower, the boundary kernel, the exact $D=0$ activities, and the preserving bond/diagonal conditions.  The numerical work is therefore organized around the missing corroboration:

1. certify the finite-size Type-I boundary kernel;
2. compare the three bounded channels on the same bond/site:
   $Q^{\rm in}=A^\dagger A$, $(Z^{\rm red})^2$, and $Y^2$;
3. verify the exact fixed-$M$ activity of the newly retained **directed transfer witness**;
4. test microcanonical activity and the surrounding spectrum at finite $D$;
5. repeat the activity test with spatially inhomogeneous $D_r$ and bond disorder;
6. evaluate the predictive deformation profile: obstruction rank and singular values, $\Delta_{\rm cage}$, the local dark-channel gap $\Delta_Q$, and the thermal pair $(\tau_Q,\chi_Q)$.

The default sizes are deliberately modest.  Increase `SIZES`, `L_STABILITY`, and the window counts locally after the workflow and symmetry resolution have been checked.

## Imports and reproducibility settings

In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.linalg as la

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from qlinks.basis.configs import basis_configs_from_build_result
from qlinks.caging import (
    LocalWitnessTemplate,
    adjacent_gap_ratio_report,
    basis_permutation_from_variable_permutation,
    cage_compatibility_hierarchy_from_hamiltonians,
    cage_jacobian_conditioning_from_hamiltonian,
    cyclic_symmetry_sector_basis,
    diagnose_cage_stability,
    diagnose_eigenpair,
    diagnose_local_channel_spectrum,
    directed_transition_witness_template,
    eigenstate_expectations,
    evaluate_local_witness_on_diagonal_ensemble,
    evaluate_local_witness_on_states,
    hermitianize_local_witness_template,
    linearized_cage_obstruction,
    permutation_matrix,
    project_operator_to_sector,
    project_state_to_sector,
    refine_sector_by_involution,
    select_microcanonical_window_by_count,
    thermal_activity_margin_from_samples,
)
from qlinks.models import (
    SpinOneXYChainModel,
    spin_one_xy_periodic_range_couplings,
    spin_one_xy_phase_compatibility,
    spin_one_xy_scar_tower_states,
    spin_one_xy_tower_thermal_activities,
)
from helpers import set_revtex_matplotlib_style

DATA_DIR = ROOT / "experimental" / "data" / "spin1_xy_draft_evidence"
DATA_DIR.mkdir(parents=True, exist_ok=True)

TOL = 1.0e-10
SIZES = (6, 8, 10)
TOTAL_SZ = -2
J_DRAFT = 1.0
J1_MATRIX = 2.0 * J_DRAFT  # qlinks matrix element; draft bond action is 2J
J3_MATRIX = 0.74
D_THERMAL = 0.63
WINDOW_FRACTION = 0.20

print("data directory:", DATA_DIR)
print("sizes:", SIZES, "fixed total Sz:", TOTAL_SZ)
print("draft J:", J_DRAFT, "qlinks nearest-neighbor matrix element:", J1_MATRIX)

set_revtex_matplotlib_style(base_font_size=12)


## Local channels and symmetry-sector helpers

In [ ]:
def make_spin1_witnesses(*, xy_matrix_element: float = J1_MATRIX):
    # Y_r=(Sz_r)^2-1 is represented on its only nonzero local pattern |0>.
    y_template = LocalWitnessTemplate(
        pattern_key=(),
        local_patterns=((0,),),
        local_operator=np.asarray([[-1.0]], dtype=np.complex128),
        metadata={"name": "Y_r", "support_sites": 1, "channel_type": "diagonal"},
    )

    # A= c |00>(<+ -|+<- +|), with c=2J in the manuscript convention.
    a_template = directed_transition_witness_template(
        target_pattern=(0, 0),
        source_patterns=((1, -1), (-1, 1)),
        amplitudes=(xy_matrix_element, xy_matrix_element),
        metadata={"name": "Ared_r_r+1", "support_sites": 2},
    )
    z_template = hermitianize_local_witness_template(
        a_template,
        metadata={"name": "Zred_r_r+1", "support_sites": 2},
    )

    raw = {
        "Y": y_template.instantiate((0,)),
        "A": a_template.instantiate((0, 1)),
        "Z": z_template.instantiate((0, 1)),
    }
    normalized = {
        name: witness.template.normalized("operator_norm").instantiate(witness.variable_indices)
        for name, witness in raw.items()
    }
    return raw, normalized


def tower_state_for_sector(basis_configs: np.ndarray, *, length: int) -> np.ndarray:
    states, labels = spin_one_xy_scar_tower_states(
        basis_configs=basis_configs,
        length=length,
        normalize=True,
    )
    if states.shape[1] != 1:
        raise RuntimeError(f"expected one tower state in a fixed-M basis, found {labels}")
    return states[:, 0]


def tower_symmetry_sector(basis_configs: np.ndarray, scar: np.ndarray, *, length: int):
    n_raised = (TOTAL_SZ + length) // 2
    momentum_index = 0 if n_raised % 2 == 0 else length // 2

    translation = basis_permutation_from_variable_permutation(
        basis_configs,
        np.roll(np.arange(length), 1),
    )
    sector = cyclic_symmetry_sector_basis(
        translation,
        order=length,
        momentum_index=momentum_index,
        labels={"total_sz": TOTAL_SZ},
    )

    # Reflection r -> -r.  k=0 and k=pi sectors are invariant under reflection.
    reflection = basis_permutation_from_variable_permutation(
        basis_configs,
        (-np.arange(length)) % length,
    )
    reflection_value = complex(np.vdot(scar, permutation_matrix(reflection) @ scar))
    reflection_parity = 1 if reflection_value.real >= 0.0 else -1
    sector = refine_sector_by_involution(
        sector,
        reflection,
        eigenvalue=reflection_parity,
        label="reflection",
    )
    return sector, momentum_index, reflection_parity


def projected_witness_square(witness, basis_configs, sector):
    local_operator = witness.embed(basis_configs)
    q_operator = local_operator.conj().T @ local_operator
    return project_operator_to_sector(q_operator, sector)


def periodic_phase_compatible_model(*, length: int, d_z: float):
    return SpinOneXYChainModel(
        length=length,
        boundary_condition="periodic",
        j_xy=J1_MATRIX,
        d_z=d_z,
        total_sz=TOTAL_SZ,
        extra_xy_couplings=spin_one_xy_periodic_range_couplings(
            length=length,
            distance=3,
            coefficient=J3_MATRIX,
        ),
    )


RAW_WITNESSES, UNIT_WITNESSES = make_spin1_witnesses()
Y_WITNESS = RAW_WITNESSES["Y"]
A_WITNESS = RAW_WITNESSES["A"]
Z_WITNESS = RAW_WITNESSES["Z"]
Y_UNIT = UNIT_WITNESSES["Y"]
A_UNIT = UNIT_WITNESSES["A"]
Z_UNIT = UNIT_WITNESSES["Z"]

witness_norm_df = pd.DataFrame(
    [
        {
            "witness": name,
            "operator_norm_raw": RAW_WITNESSES[name].template.operator_norm,
            "Q_norm_raw": RAW_WITNESSES[name].template.q_operator_norm,
            "Delta_Q_unit": diagnose_local_channel_spectrum(
                UNIT_WITNESSES[name],
                tolerance=TOL,
            ).dark_channel_gap,
            "Q_rank_unit": diagnose_local_channel_spectrum(
                UNIT_WITNESSES[name],
                tolerance=TOL,
            ).rank,
        }
        for name in ("A", "Z", "Y")
    ]
)
display(witness_norm_df)

## A. Exact Type-I boundary kernel and three bounded local channels

In [ ]:
L_REP = 8
model_rep = periodic_phase_compatible_model(length=L_REP, d_z=0.0)
build_rep = model_rep.build(
    builder="optimized",
    basis_solver="dfs",
    sort_basis=True,
    on_missing="raise",
)
configs_rep = basis_configs_from_build_result(build_rep)
scar_rep = tower_state_for_sector(configs_rep, length=L_REP)
support_rep = np.flatnonzero(np.abs(scar_rep) > TOL)

stability_rep = diagnose_cage_stability(
    build_rep.kinetic,
    support_rep,
    state=scar_rep,
    tolerance=TOL,
)
eigenpair_rep = diagnose_eigenpair(build_rep.hamiltonian, scar_rep)
witness_evaluations = {
    name: evaluate_local_witness_on_states(
        witness,
        basis_configs=configs_rep,
        states=scar_rep,
    )
    for name, witness in RAW_WITNESSES.items()
}

local_rows = []
for descriptor in model_rep.local_term_descriptors(operator_kind="kinetic", term_kind="bond"):
    local_matrix = model_rep.build_local_term(descriptor, build_rep, builder="optimized")
    local_rows.append(
        {
            "term": descriptor.label,
            "sites": descriptor.support_sites,
            "action_norm": float(np.linalg.norm(local_matrix @ scar_rep)),
        }
    )
local_term_df = pd.DataFrame(local_rows)

boundary_scorecard = pd.DataFrame(
    [
        {
            "L": L_REP,
            "M": TOTAL_SZ,
            "full_sector_dimension": configs_rep.shape[0],
            "support_size": support_rep.size,
            "boundary_rank": stability_rep.boundary_rank,
            "boundary_nullity": stability_rep.boundary_nullity,
            "boundary_singular_gap": stability_rep.interference_gap,
            "boundary_residual": stability_rep.state_boundary_residual,
            "internal_residual": stability_rep.state_internal_eigen_residual,
            "full_eigenpair_residual": eigenpair_rep.residual_norm,
            "A_annihilation_residual": witness_evaluations["A"].annihilation_residual,
            "Z_annihilation_residual": witness_evaluations["Z"].annihilation_residual,
            "Y_annihilation_residual": witness_evaluations["Y"].annihilation_residual,
        }
    ]
)

display(boundary_scorecard)
display(local_term_df)
boundary_scorecard.to_csv(DATA_DIR / "boundary_kernel_scorecard.csv", index=False)
local_term_df.to_csv(DATA_DIR / "local_term_annihilation.csv", index=False)
witness_norm_df.to_csv(DATA_DIR / "local_channel_spectra.csv", index=False)

The representative certificate separates four facts:

- the shell support has size $\binom{L}{n}$ and the boundary nullity is one;
- the full eigenpair and every bond action have vanishing residual;
- the directed map $A$, its Hermitianization $Z$, and the diagonal $Y$ all annihilate the tower;
- after operator-norm normalization, the smallest positive eigenvalue $\Delta_Q$ of each local positive channel is reported separately from exact compatibility.

## B. Exact finite-$L$ thermal activities of $Q^{\rm in}$, $Z^2$, and $Y^2$

The exact fixed-$M$ formulas now come with **explicit thermodynamic asymptote lines**. For the present sequence $M=-2$ and $L\to\infty$, the magnetization density tends to $q_\infty=0$, so $p_0\to 1/3$.  The three witness activities therefore approach

$$\lim_{L\to\infty}\operatorname{Tr}(\rho_{\infty,M}Y_r^2)=\frac13,$$
$$\lim_{L\to\infty}\operatorname{Tr}(\rho_{\infty,M}A_r^\dagger A_r)=2|2J|^2\left(\frac13\right)^2,$$
$$\lim_{L\to\infty}\operatorname{Tr}(\rho_{\infty,M}Z_r^2)=4|2J|^2\left(\frac13\right)^2,$$

with the last two divided by the corresponding operator norms in the normalized comparison plot.

In [ ]:
formula_rows = []
for length in range(4, 32, 2):
    exact = spin_one_xy_tower_thermal_activities(
        length=length,
        total_sz=TOTAL_SZ,
        xy_matrix_element=J1_MATRIX,
    )
    formula_rows.append(exact.to_summary_dict())
formula_df = pd.DataFrame(formula_rows)

# Independent direct traces in the qlinks fixed-M basis for ED-accessible sizes.
direct_rows = []
for length in (4, 6, 8, 10):
    result = SpinOneXYChainModel(
        length=length,
        boundary_condition="periodic",
        j_xy=0.0,
        total_sz=TOTAL_SZ,
    ).build(builder="optimized", basis_solver="dfs", sort_basis=True)
    configs = basis_configs_from_build_result(result)
    evaluations = {
        name: evaluate_local_witness_on_diagonal_ensemble(
            witness,
            basis_configs=configs,
        )
        for name, witness in RAW_WITNESSES.items()
    }
    direct_rows.append(
        {
            "length": length,
            "basis_dimension": configs.shape[0],
            "Y2_direct_trace": evaluations["Y"].expectation,
            "A2_direct_trace": evaluations["A"].expectation,
            "Z2_direct_trace": evaluations["Z"].expectation,
            "A2_direct_normalized": evaluations["A"].normalized_expectation,
            "Z2_direct_normalized": evaluations["Z"].normalized_expectation,
        }
    )
direct_df = pd.DataFrame(direct_rows)
activity_df = formula_df.merge(direct_df, how="left", on="length")
activity_df["Y2_direct_minus_formula"] = activity_df["Y2_direct_trace"] - activity_df["y2_activity"]
activity_df["A2_direct_minus_formula"] = activity_df["A2_direct_trace"] - activity_df["directed_q_activity"]
activity_df["Z2_direct_minus_formula"] = activity_df["Z2_direct_trace"] - activity_df["z2_activity"]

display(activity_df.head(8))
activity_df.to_csv(DATA_DIR / "exact_fixed_M_activities.csv", index=False)

fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(activity_df["length"], activity_df["y2_activity"], marker="o", label=r"$Y_r^2$")
ax.plot(
    activity_df["length"],
    activity_df["directed_q_activity"] / A_WITNESS.template.q_operator_norm,
    marker="s",
    label=r"$\widetilde A^\dagger\widetilde A$",
)
ax.plot(
    activity_df["length"],
    activity_df["z2_activity"] / Z_WITNESS.template.q_operator_norm,
    marker="^",
    label=r"$(\widetilde Z^{\rm red})^2$",
)
ax.set_xlabel(r"$L$")
ax.set_ylabel("operator-norm-normalized thermal activity")
ax.legend(loc="upper right")
ax.grid()
fig.tight_layout()
fig.savefig(DATA_DIR / "exact_fixed_M_three_witness_activities.pdf")

# Thermodynamic asymptotes for the fixed-M sequence TOTAL_SZ=-2, where q=M/L -> 0.
p0_infty = 1.0 / 3.0
y2_infty = p0_infty
a2_infty = 2.0 * abs(J1_MATRIX) ** 2 * p0_infty**2
z2_infty = 4.0 * abs(J1_MATRIX) ** 2 * p0_infty**2
a2_infty_normalized = a2_infty / A_WITNESS.template.q_operator_norm
z2_infty_normalized = z2_infty / Z_WITNESS.template.q_operator_norm

fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(activity_df["length"], activity_df["y2_activity"], marker="o", label=r"$Y_r^2$")
ax.plot(activity_df["length"], activity_df["directed_q_activity"], marker="s", label=r"$A_r^\dagger A_r$")
ax.plot(activity_df["length"], activity_df["z2_activity"], marker="^", label=r"$Z_r^2$")
ax.axhline(y2_infty, linestyle="--", label=r"$Y_r^2$ asymptote")
ax.axhline(a2_infty, linestyle=":", label=r"$A_r^\dagger A_r$ asymptote")
ax.axhline(z2_infty, linestyle="-.", label=r"$Z_r^2$ asymptote")
ax.set_xlabel(r"$L$")
ax.set_ylabel("raw fixed-$M$ thermal activity")
ax.legend(loc="best", fontsize=9)
ax.grid()
fig.tight_layout()
fig.savefig(DATA_DIR / "exact_fixed_M_three_witness_activities_raw_with_asymptotes.pdf")

fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(activity_df["length"], activity_df["y2_activity"], marker="o", label=r"$Y_r^2$")
ax.plot(
    activity_df["length"],
    activity_df["directed_q_activity"] / A_WITNESS.template.q_operator_norm,
    marker="s",
    label=r"$\widetilde A^\dagger\widetilde A$",
)
ax.plot(
    activity_df["length"],
    activity_df["z2_activity"] / Z_WITNESS.template.q_operator_norm,
    marker="^",
    label=r"$(\widetilde Z^{\rm red})^2$",
)
ax.axhline(y2_infty, linestyle="--", label=r"$Y_r^2$ asymptote")
ax.axhline(a2_infty_normalized, linestyle=":", label=r"$\widetilde A^\dagger\widetilde A$ asymptote")
ax.axhline(z2_infty_normalized, linestyle="-.", label=r"$(\widetilde Z^{\rm red})^2$ asymptote")
ax.set_xlabel(r"$L$")
ax.set_ylabel("operator-norm-normalized thermal activity")
ax.legend(loc="best", fontsize=9)
ax.grid()
fig.tight_layout()
fig.savefig(DATA_DIR / "exact_fixed_M_three_witness_activities_with_asymptotes.pdf")

asymptote_df = pd.DataFrame(
    [
        {
            "sequence": "fixed_M_to_q0",
            "p0_infty": p0_infty,
            "Y2_asymptote": y2_infty,
            "A2_asymptote": a2_infty,
            "A2_asymptote_normalized": a2_infty_normalized,
            "Z2_asymptote": z2_infty,
            "Z2_asymptote_normalized": z2_infty_normalized,
        }
    ]
)
display(asymptote_df)
asymptote_df.to_csv(DATA_DIR / "exact_fixed_M_activities_asymptotes.csv", index=False)


## C. Symmetry-resolved microcanonical activities, ETH scatter, and level statistics

In [ ]:
spectral_rows = []
scan_cache = {}

for length in SIZES:
    t0 = time.perf_counter()
    n_raised = (TOTAL_SZ + length) // 2

    result_zero = periodic_phase_compatible_model(length=length, d_z=0.0).build(
        builder="optimized",
        basis_solver="dfs",
        sort_basis=True,
    )
    configs = basis_configs_from_build_result(result_zero)
    scar = tower_state_for_sector(configs, length=length)
    sector, momentum_index, reflection_parity = tower_symmetry_sector(
        configs,
        scar,
        length=length,
    )
    scar_sector = project_state_to_sector(scar, sector)
    qy_sector = projected_witness_square(Y_WITNESS, configs, sector)
    qa_sector = projected_witness_square(A_WITNESS, configs, sector)
    qz_sector = projected_witness_square(Z_WITNESS, configs, sector)

    h0_sector = project_operator_to_sector(result_zero.hamiltonian, sector)
    e0, v0 = la.eigh(h0_sector)
    y0 = eigenstate_expectations(qy_sector, v0)
    a0 = eigenstate_expectations(qa_sector, v0)
    z0 = eigenstate_expectations(qz_sector, v0)
    target_count = min(
        sector.sector_dimension,
        max(10, int(math.ceil(WINDOW_FRACTION * sector.sector_dimension))),
    )
    window0 = select_microcanonical_window_by_count(
        e0,
        target_energy=0.0,
        target_count=target_count,
        include_boundary_degeneracy=True,
    )
    idx0 = np.asarray(window0.indices, dtype=np.int64)

    result_d = periodic_phase_compatible_model(length=length, d_z=D_THERMAL).build(
        builder="optimized",
        basis_solver="dfs",
        sort_basis=True,
    )
    np.testing.assert_array_equal(result_d.basis.states, result_zero.basis.states)
    hd_sector = project_operator_to_sector(result_d.hamiltonian, sector)
    ed, vd = la.eigh(hd_sector)
    yd = eigenstate_expectations(qy_sector, vd)
    ad = eigenstate_expectations(qa_sector, vd)
    zd = eigenstate_expectations(qz_sector, vd)
    scar_energy = D_THERMAL * length
    overlap = np.abs(vd.conj().T @ scar_sector)
    scar_level = int(np.argmax(overlap))
    windowd = select_microcanonical_window_by_count(
        ed,
        target_energy=scar_energy,
        target_count=target_count,
        include_boundary_degeneracy=True,
    )
    idxd = np.asarray(windowd.indices, dtype=np.int64)
    gap = adjacent_gap_ratio_report(
        ed,
        trim_fraction=0.10,
        degeneracy_tolerance=1.0e-8,
    )
    exact = spin_one_xy_tower_thermal_activities(
        length=length,
        total_sz=TOTAL_SZ,
        xy_matrix_element=J1_MATRIX,
    )
    residual_d = diagnose_eigenpair(result_d.hamiltonian, scar)

    spectral_rows.append(
        {
            "L": length,
            "M": TOTAL_SZ,
            "n_raised": n_raised,
            "full_M_sector_dimension": configs.shape[0],
            "momentum_index": momentum_index,
            "momentum_over_pi": 2.0 * momentum_index / length,
            "reflection_parity": reflection_parity,
            "resolved_sector_dimension": sector.sector_dimension,
            "D": D_THERMAL,
            "scar_energy": scar_energy,
            "scar_level_energy": ed[scar_level],
            "scar_overlap": overlap[scar_level],
            "scar_residual": residual_d.residual_norm,
            "D0_window_half_width": window0.half_width,
            "D0_window_state_count": window0.n_states,
            "D0_window_center_offset": window0.center_offset,
            "D0_microcanonical_Y2": float(np.mean(y0[idx0])),
            "D0_microcanonical_A2": float(np.mean(a0[idx0])),
            "D0_microcanonical_Z2": float(np.mean(z0[idx0])),
            "exact_fixed_M_Y2": exact.y2_activity,
            "exact_fixed_M_A2": exact.directed_q_activity,
            "exact_fixed_M_Z2": exact.z2_activity,
            "finiteD_window_half_width": windowd.half_width,
            "finiteD_window_state_count": windowd.n_states,
            "finiteD_window_center_offset": windowd.center_offset,
            "finiteD_microcanonical_Y2": float(np.mean(yd[idxd])),
            "finiteD_microcanonical_A2": float(np.mean(ad[idxd])),
            "finiteD_microcanonical_Z2": float(np.mean(zd[idxd])),
            "finiteD_unit_A_activity": float(np.mean(ad[idxd]) / A_WITNESS.template.q_operator_norm),
            "mean_gap_ratio": gap.mean_ratio,
            "gap_ratio_count": len(gap.ratios),
            "runtime_seconds": time.perf_counter() - t0,
        }
    )
    scan_cache[length] = {
        "configs": configs,
        "scar": scar,
        "sector": sector,
        "energies_D": ed,
        "vectors_D": vd,
        "Y2_D": yd,
        "A2_D": ad,
        "Z2_D": zd,
        "scar_level": scar_level,
        "window_D": windowd,
        "gap_report": gap,
    }

spectral_df = pd.DataFrame(spectral_rows)
display(spectral_df)
spectral_df.to_csv(DATA_DIR / "symmetry_resolved_spectral_evidence.csv", index=False)

fig, ax = plt.subplots(figsize=(6.4, 4.0))
for column, exact_column, marker, label in (
    ("D0_microcanonical_Y2", "exact_fixed_M_Y2", "o", r"$Y_r^2$"),
    ("D0_microcanonical_A2", "exact_fixed_M_A2", "s", r"$A^\dagger A$"),
    ("D0_microcanonical_Z2", "exact_fixed_M_Z2", "^", r"$(Z^{\rm red})^2$"),
):
    ax.plot(
        spectral_df["L"],
        spectral_df[column] / spectral_df[exact_column],
        marker=marker,
        label=label,
    )
ax.axhline(1.0, linestyle=":")
ax.set_xlabel(r"$L$")
ax.set_ylabel("microcanonical / exact fixed-$M$ activity")
ax.legend()
fig.tight_layout()

fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(spectral_df["L"], spectral_df["mean_gap_ratio"], marker="o", label="resolved data")
ax.axhline(0.5307, linestyle="--", label="GOE")
ax.axhline(2.0 * np.log(2.0) - 1.0, linestyle=":", label="Poisson")
ax.set_xlabel(r"$L$")
ax.set_ylabel(r"mean adjacent-gap ratio $\langle r\rangle$")
ax.legend()
fig.tight_layout()

The $D=0$ microcanonical comparison must be interpreted with care: chiral symmetry produces a finite zero-mode manifold in the resolved sector. The window selection includes complete degeneracies at its boundary. The finite-$D$ data lift this accidental zero-mode degeneracy while leaving the tower exact, and are therefore the cleaner level-statistics benchmark.

### ETH scatter at the largest resolved size

In [ ]:
largest = scan_cache[max(SIZES)]
energies = largest["energies_D"]
y_values = largest["Y2_D"]
a_values = largest["A2_D"]
z_values = largest["Z2_D"]
scar_level = largest["scar_level"]

scatter_df = pd.DataFrame(
    {
        "energy": energies,
        "energy_density": energies / max(SIZES),
        "Y2": y_values,
        "A2": a_values,
        "Z2": z_values,
        "is_scar_level": np.arange(energies.size) == scar_level,
    }
)
scatter_df.to_csv(DATA_DIR / "eth_scatter_Lmax_finite_D.csv", index=False)

fig, ax = plt.subplots(figsize=(6.8, 4.2))
ax.scatter(scatter_df["energy_density"], scatter_df["Y2"], s=12, alpha=0.65, label=r"$Y_r^2$")
ax.scatter(scatter_df["energy_density"], scatter_df["A2"], s=12, alpha=0.65, label=r"$A^\dagger A$")
ax.scatter(scatter_df["energy_density"], scatter_df["Z2"], s=12, alpha=0.65, label=r"$(Z^{\rm red})^2$")
ax.scatter(
    [scatter_df.loc[scar_level, "energy_density"]],
    [0.0],
    marker="*",
    color='green',
    s=150,
    label=r"exact tower state",
)
ax.set_xlabel(r"energy density $e$")
ax.set_ylabel(r"local positive-witness activity")
ax.legend(loc="upper right")
ax.grid()
fig.tight_layout()

display(scatter_df.iloc[max(0, scar_level - 3): scar_level + 4])

In [ ]:
fig.savefig(
    "../images/spin1_xy_eth_scatter.pdf",
    bbox_inches="tight",
    pad_inches=0.02,
)

## D. Bondwise deformation condition and controlled violation

In [ ]:
L_DEF = 6
phases = (-1.0) ** np.arange(L_DEF)
nearest_pairs = spin_one_xy_periodic_range_couplings(
    length=L_DEF,
    distance=1,
    coefficient=J1_MATRIX,
)
third_pairs = spin_one_xy_periodic_range_couplings(
    length=L_DEF,
    distance=3,
    coefficient=J3_MATRIX,
)
compatibility = spin_one_xy_phase_compatibility(
    nearest_pairs + third_pairs,
    phases=phases,
)
compatibility_df = pd.DataFrame(
    [
        {
            "site_i": pair[0],
            "site_j": pair[1],
            "coupling": coupling,
            "phase_condition_residual": residual,
            "absolute_residual": abs(residual),
        }
        for pair, coupling, residual in zip(
            compatibility.pairs,
            compatibility.couplings,
            compatibility.residuals,
            strict=True,
        )
    ]
)
display(compatibility_df)
compatibility_df.to_csv(DATA_DIR / "bondwise_phase_compatibility.csv", index=False)

base_result = periodic_phase_compatible_model(length=L_DEF, d_z=D_THERMAL).build(
    builder="optimized",
    basis_solver="dfs",
    sort_basis=True,
)
base_configs = basis_configs_from_build_result(base_result)
base_scar = tower_state_for_sector(base_configs, length=L_DEF)
violating_result = SpinOneXYChainModel(
    length=L_DEF,
    boundary_condition="periodic",
    j_xy=0.0,
    total_sz=TOTAL_SZ,
    extra_xy_couplings=((0, 2, 1.0),),  # same-sublattice exchange violates Eq. (134)
).build(builder="optimized", basis_solver="dfs", sort_basis=True)
np.testing.assert_array_equal(violating_result.basis.states, base_result.basis.states)

violation_rows = []
for epsilon in np.linspace(0.0, 0.20, 11):
    hamiltonian = base_result.hamiltonian + epsilon * violating_result.hamiltonian
    report = diagnose_eigenpair(hamiltonian, base_scar)
    phase_report = spin_one_xy_phase_compatibility(
        nearest_pairs + third_pairs + ((0, 2, epsilon),),
        phases=phases,
    )
    violation_rows.append(
        {
            "epsilon": epsilon,
            "max_phase_condition_residual": phase_report.max_residual,
            "scar_residual": report.residual_norm,
            "scar_variance": report.variance,
        }
    )
violation_df = pd.DataFrame(violation_rows)
display(violation_df)
violation_df.to_csv(DATA_DIR / "phase_condition_violation.csv", index=False)

fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(violation_df["epsilon"], violation_df["scar_residual"], marker="o")
ax.set_xlabel(r"phase-incompatible coupling $\epsilon$")
ax.set_ylabel(r"$\|(H-E)|S_n\rangle\|$")
ax.grid()
fig.tight_layout()
ax.set_yscale("log")
fig.savefig(DATA_DIR / "phase_condition_violation_residual.pdf")

fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(violation_df["epsilon"], violation_df["max_phase_condition_residual"], marker="o")
ax.set_xlabel(r"phase-incompatible coupling $\epsilon$")
ax.set_ylabel("max bondwise phase-condition residual")
ax.grid()
fig.tight_layout()
fig.savefig(DATA_DIR / "phase_condition_violation_obstruction.pdf")


## E. Inhomogeneous single-ion anisotropy with a phase-compatible bond-disordered thermal background

In [ ]:
L_INHOM = 6
rng = np.random.default_rng(13)
sites = np.arange(L_INHOM)

# Arbitrary real exchanges between opposite sublattices satisfy Eq. (134) for eta_r=(-1)^r.
# Random bond strengths break translation and reflection while preserving the tower exactly.
inhom_pairs = []
for site_i, site_j, _ in spin_one_xy_periodic_range_couplings(
    length=L_INHOM,
    distance=1,
    coefficient=1.0,
):
    inhom_pairs.append((site_i, site_j, float(1.5 + 0.8 * rng.random())))
for site_i, site_j, _ in spin_one_xy_periodic_range_couplings(
    length=L_INHOM,
    distance=3,
    coefficient=1.0,
):
    inhom_pairs.append((site_i, site_j, float(0.2 + 0.8 * rng.random())))

d_profile = 0.4 + 0.4 * rng.random(L_INHOM)
inhom_phase = spin_one_xy_phase_compatibility(
    tuple(inhom_pairs),
    phases=(-1.0) ** sites,
)
assert inhom_phase.is_compatible

inhom_model = SpinOneXYChainModel(
    length=L_INHOM,
    boundary_condition="periodic",
    j_xy=0.0,
    d_z_by_site=tuple(float(value) for value in d_profile),
    total_sz=TOTAL_SZ,
    extra_xy_couplings=tuple(inhom_pairs),
)
inhom_result = inhom_model.build(
    builder="optimized",
    basis_solver="dfs",
    sort_basis=True,
)
inhom_configs = basis_configs_from_build_result(inhom_result)
inhom_scar = tower_state_for_sector(inhom_configs, length=L_INHOM)
inhom_residual = diagnose_eigenpair(inhom_result.hamiltonian, inhom_scar)
inhom_scar_energy = float(np.sum(d_profile))

# Spatial symmetries are deliberately broken, so the fixed-M block is already desymmetrized.
inhom_h = inhom_result.hamiltonian.toarray()
inhom_energies, inhom_vectors = la.eigh(inhom_h)
y_local = Y_WITNESS.embed(inhom_configs)
a_local = A_WITNESS.embed(inhom_configs)
z_local = Z_WITNESS.embed(inhom_configs)
y2_inhom = eigenstate_expectations(y_local.conj().T @ y_local, inhom_vectors)
a2_inhom = eigenstate_expectations(a_local.conj().T @ a_local, inhom_vectors)
z2_inhom = eigenstate_expectations(z_local.conj().T @ z_local, inhom_vectors)
inhom_overlap = np.abs(inhom_vectors.conj().T @ inhom_scar)
inhom_scar_level = int(np.argmax(inhom_overlap))
inhom_window = select_microcanonical_window_by_count(
    inhom_energies,
    target_energy=inhom_scar_energy,
    target_count=80,
    include_boundary_degeneracy=True,
)
inhom_indices = np.asarray(inhom_window.indices, dtype=np.int64)
inhom_gap = adjacent_gap_ratio_report(
    inhom_energies,
    trim_fraction=0.10,
    degeneracy_tolerance=1.0e-8,
)

inhom_df = pd.DataFrame(
    [
        {
            "L": L_INHOM,
            "M": TOTAL_SZ,
            "full_sector_dimension": inhom_configs.shape[0],
            "max_phase_condition_residual": inhom_phase.max_residual,
            "scar_energy_expected": inhom_scar_energy,
            "scar_energy_eigensolver": inhom_energies[inhom_scar_level],
            "scar_overlap": inhom_overlap[inhom_scar_level],
            "scar_residual": inhom_residual.residual_norm,
            "window_half_width": inhom_window.half_width,
            "window_state_count": inhom_window.n_states,
            "window_center_offset": inhom_window.center_offset,
            "microcanonical_Y2": float(np.mean(y2_inhom[inhom_indices])),
            "microcanonical_A2": float(np.mean(a2_inhom[inhom_indices])),
            "microcanonical_unit_A": float(np.mean(a2_inhom[inhom_indices]) / A_WITNESS.template.q_operator_norm),
            "microcanonical_Z2": float(np.mean(z2_inhom[inhom_indices])),
            "mean_gap_ratio": inhom_gap.mean_ratio,
            "gap_ratio_count": len(inhom_gap.ratios),
        }
    ]
)
inhom_profile_df = pd.DataFrame({"site": sites, "D_r": d_profile})
inhom_coupling_df = pd.DataFrame(
    [
        {
            "site_i": site_i,
            "site_j": site_j,
            "matrix_element": coupling,
        }
        for site_i, site_j, coupling in inhom_pairs
    ]
)
display(inhom_profile_df)
display(inhom_coupling_df)
display(inhom_df)
inhom_profile_df.to_csv(DATA_DIR / "inhomogeneous_D_profile.csv", index=False)
inhom_coupling_df.to_csv(DATA_DIR / "inhomogeneous_phase_compatible_couplings.csv", index=False)
inhom_df.to_csv(DATA_DIR / "inhomogeneous_D_evidence.csv", index=False)

## F. Predictive deformation profile

The draft criterion distinguishes **compatibility** from **conditioning**.  We therefore report:

- the full caged-eigenpair obstruction map for several normalized local coupling alphabets;
- its rank, compatible dimension, and nonzero singular spectrum;
- the gauge-fixed cage-Jacobian gap $\Delta_{\rm cage}$;
- the directed local-channel obstruction and $\Delta_Q$;
- the finite-size thermal activity $\tau_Q$ and a conservative secant estimate of $\chi_Q$.

These are finite-size data.  A thermodynamic stability claim additionally requires uniform bounds with increasing $L$.

In [ ]:
L_STABILITY = 6
base_stability_model = periodic_phase_compatible_model(length=L_STABILITY, d_z=D_THERMAL)
base_stability = base_stability_model.build(
    builder="optimized",
    basis_solver="dfs",
    sort_basis=True,
)
stability_configs = basis_configs_from_build_result(base_stability)
stability_scar = tower_state_for_sector(stability_configs, length=L_STABILITY)
stability_support = np.flatnonzero(np.abs(stability_scar) > TOL)


def perturbation_matrix(*, pairs=(), d_profile=None, h_profile=None):
    model = SpinOneXYChainModel(
        length=L_STABILITY,
        boundary_condition="periodic",
        j_xy=0.0,
        total_sz=TOTAL_SZ,
        extra_xy_couplings=tuple(pairs),
        d_z_by_site=None if d_profile is None else tuple(complex(x) for x in d_profile),
        h_z_by_site=None if h_profile is None else tuple(complex(x) for x in h_profile),
    )
    result = model.build(builder="optimized", basis_solver="dfs", sort_basis=True)
    np.testing.assert_array_equal(result.basis.states, base_stability.basis.states)
    return result.hamiltonian


nearest_unit = spin_one_xy_periodic_range_couplings(
    length=L_STABILITY,
    distance=1,
    coefficient=1.0,
)
third_unit = spin_one_xy_periodic_range_couplings(
    length=L_STABILITY,
    distance=3,
    coefficient=1.0,
)
second_unit = spin_one_xy_periodic_range_couplings(
    length=L_STABILITY,
    distance=2,
    coefficient=1.0,
)

def one_hot(site):
    return tuple(1.0 if index == site else 0.0 for index in range(L_STABILITY))

alphabets = {
    "odd_range_real": [
        perturbation_matrix(pairs=((i, j, coefficient),))
        for i, j, coefficient in (*nearest_unit, *third_unit)
    ],
    "inhomogeneous_Dr": [
        perturbation_matrix(d_profile=one_hot(site))
        for site in range(L_STABILITY)
    ],
    "inhomogeneous_hr": [
        perturbation_matrix(h_profile=one_hot(site))
        for site in range(L_STABILITY)
    ],
    "even_range_real": [
        perturbation_matrix(pairs=((i, j, coefficient),))
        for i, j, coefficient in second_unit
    ],
}

obstruction_rows = []
obstruction_spectra = []
for alphabet_name, perturbations in alphabets.items():
    hierarchy = cage_compatibility_hierarchy_from_hamiltonians(
        base_stability.hamiltonian,
        perturbations,
        stability_support,
        stability_scar,
        coefficient_field="real",
        tolerance=TOL,
    )
    first_order = hierarchy.first_order
    obstruction_rows.append(
        {
            "alphabet": alphabet_name,
            "n_parameters": first_order.n_parameters,
            "obstruction_rank": first_order.rank,
            "first_order_compatible_dimension": first_order.compatible_dimension,
            "fixed_state_compatible_dimension": hierarchy.fixed_state.compatible_dimension,
            "tangent_only_dimension": hierarchy.tangent_only_dimension,
        }
    )
    for index, value in enumerate(first_order.singular_values):
        obstruction_spectra.append(
            {
                "alphabet": alphabet_name,
                "singular_index": index,
                "singular_value": float(value),
            }
        )

obstruction_df = pd.DataFrame(obstruction_rows)
obstruction_spectrum_df = pd.DataFrame(obstruction_spectra)
cage_conditioning = cage_jacobian_conditioning_from_hamiltonian(
    base_stability.hamiltonian,
    stability_support,
    stability_scar,
    tolerance=TOL,
)

display(obstruction_df)
display(pd.DataFrame([cage_conditioning.to_summary_dict()]))
obstruction_df.to_csv(DATA_DIR / "deformation_obstruction_scorecard.csv", index=False)
obstruction_spectrum_df.to_csv(DATA_DIR / "deformation_obstruction_spectra.csv", index=False)
pd.DataFrame([cage_conditioning.to_summary_dict()]).to_csv(
    DATA_DIR / "cage_jacobian_conditioning.csv",
    index=False,
)

# Draft-oriented deformation figures.
scorecard_plot_df = obstruction_df.copy()
scorecard_plot_df = scorecard_plot_df.sort_values(
    ["first_order_compatible_dimension", "n_parameters"],
    ascending=[False, True],
).reset_index(drop=True)
scorecard_plot_df["obstructed_dimension"] = (
    scorecard_plot_df["n_parameters"] - scorecard_plot_df["first_order_compatible_dimension"]
)
scorecard_plot_df["floating_compatible_dimension"] = scorecard_plot_df["tangent_only_dimension"]
labels = scorecard_plot_df["alphabet"].tolist()
ypos = np.arange(len(labels))

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.barh(ypos, scorecard_plot_df["first_order_compatible_dimension"], label="first-order compatible")
ax.barh(
    ypos,
    scorecard_plot_df["obstructed_dimension"],
    left=scorecard_plot_df["first_order_compatible_dimension"],
    label="obstructed",
)
ax.plot(
    scorecard_plot_df["fixed_state_compatible_dimension"],
    ypos,
    marker="o",
    linestyle="None",
    label="fixed-state compatible",
)
ax.set_yticks(ypos, labels)
ax.set_xlabel("parameter-space dimension")
ax.set_ylabel("deformation alphabet")
ax.legend(loc="best", fontsize=9)
ax.grid(axis="x")
fig.tight_layout()
fig.savefig(DATA_DIR / "deformation_obstruction_scorecard.pdf")

fig, ax = plt.subplots(figsize=(6.8, 4.0))
for alphabet, frame in obstruction_spectrum_df.groupby("alphabet", sort=False):
    ordered = frame.sort_values("singular_index")
    ax.semilogy(
        ordered["singular_index"] + 1,
        np.maximum(ordered["singular_value"], 1.0e-16),
        marker="o",
        label=alphabet,
    )
ax.set_xlabel("singular-value index")
ax.set_ylabel("first-order obstruction singular value")
ax.legend(loc="best", fontsize=8)
ax.grid()
fig.tight_layout()
fig.savefig(DATA_DIR / "deformation_obstruction_singular_spectra.pdf")


### Local directed-channel compatibility and $\Delta_Q$

The local obstruction criterion allows the conditional dark vector to rotate.  For a rank-one directed row, both a common rescaling and a small imbalance of the two incoming amplitudes remain first-order compatible because the imbalance can be absorbed by rotating the local kernel.  The **fixed** antisymmetric vector is more restrictive: it survives the common rescaling but not the imbalance.  This is a useful concrete distinction between continuation of a dark channel and preservation of one preselected tower vector.

In [ ]:
a_local_unit = np.asarray(A_UNIT.local_operator, dtype=np.complex128)
local_dark_vector = np.asarray([0.0, 1.0, -1.0], dtype=np.complex128) / np.sqrt(2.0)
scale_perturbation = a_local_unit.copy()
imbalance_perturbation = np.zeros_like(a_local_unit)
imbalance_perturbation[0, 1] = 1.0
imbalance_perturbation[0, 2] = -1.0

local_obstruction = linearized_cage_obstruction(
    a_local_unit,
    local_dark_vector,
    (scale_perturbation, imbalance_perturbation),
    coefficient_field="real",
    tolerance=TOL,
)
local_q_spectrum = diagnose_local_channel_spectrum(A_UNIT, tolerance=TOL)
local_channel_df = pd.DataFrame(
    [
        {
            "perturbation": "common_scale",
            "fixed_dark_vector_residual": float(
                np.linalg.norm(scale_perturbation @ local_dark_vector)
            ),
            "first_order_obstruction_residual": float(
                np.linalg.norm(local_obstruction.obstruction_matrix[:, 0])
            ),
        },
        {
            "perturbation": "source_imbalance",
            "fixed_dark_vector_residual": float(
                np.linalg.norm(imbalance_perturbation @ local_dark_vector)
            ),
            "first_order_obstruction_residual": float(
                np.linalg.norm(local_obstruction.obstruction_matrix[:, 1])
            ),
        },
    ]
)
local_channel_summary_df = pd.DataFrame(
    [
        {
            "obstruction_rank": local_obstruction.rank,
            "compatible_dimension": local_obstruction.compatible_dimension,
            "Delta_Q": local_q_spectrum.dark_channel_gap,
            "Q_rank": local_q_spectrum.rank,
            "Q_nullity": local_q_spectrum.nullity,
            "witness_radius_bonds": 1,
        }
    ]
)
display(local_channel_df)
display(local_channel_summary_df)
local_channel_df.to_csv(DATA_DIR / "directed_local_channel_perturbations.csv", index=False)
local_channel_summary_df.to_csv(
    DATA_DIR / "directed_local_channel_stability.csv",
    index=False,
)

### Finite-$D$ thermal margin along a compatible path

The parameter below is the Euclidean norm of the sitewise coupling vector, $g=\sqrt{L}\,(D-D_0)$, so the secant susceptibility is tied to the stated deformation alphabet normalization.

In [ ]:
D_PATH = D_THERMAL + np.linspace(-0.20, 0.20, 5)
finite_d_margin_rows = []
for d_value in D_PATH:
    model = periodic_phase_compatible_model(length=L_STABILITY, d_z=float(d_value))
    result = model.build(builder="optimized", basis_solver="dfs", sort_basis=True)
    configs = basis_configs_from_build_result(result)
    scar = tower_state_for_sector(configs, length=L_STABILITY)
    sector, _, _ = tower_symmetry_sector(configs, scar, length=L_STABILITY)
    projected_h = project_operator_to_sector(result.hamiltonian, sector)
    energies, vectors = la.eigh(projected_h)
    projected_q = projected_witness_square(A_UNIT, configs, sector)
    activities = eigenstate_expectations(projected_q, vectors)
    target_count = min(
        sector.sector_dimension,
        max(10, int(math.ceil(WINDOW_FRACTION * sector.sector_dimension))),
    )
    window = select_microcanonical_window_by_count(
        energies,
        target_energy=float(d_value * L_STABILITY),
        target_count=target_count,
        include_boundary_degeneracy=True,
    )
    indices = np.asarray(window.indices, dtype=np.int64)
    finite_d_margin_rows.append(
        {
            "D": float(d_value),
            "coupling_path_parameter": float(np.sqrt(L_STABILITY) * (d_value - D_THERMAL)),
            "scar_residual": diagnose_eigenpair(result.hamiltonian, scar).residual_norm,
            "window_state_count": window.n_states,
            "window_half_width": window.half_width,
            "unit_directed_activity": float(np.mean(activities[indices])),
        }
    )
finite_d_margin_df = pd.DataFrame(finite_d_margin_rows)
finite_d_margin = thermal_activity_margin_from_samples(
    finite_d_margin_df["coupling_path_parameter"],
    finite_d_margin_df["unit_directed_activity"],
    reference_parameter=0.0,
    tolerance=TOL,
)
display(finite_d_margin_df)
display(pd.DataFrame([finite_d_margin.to_summary_dict()]))
finite_d_margin_df.to_csv(DATA_DIR / "finite_D_directed_thermal_path.csv", index=False)
pd.DataFrame([finite_d_margin.to_summary_dict()]).to_csv(
    DATA_DIR / "finite_D_directed_thermal_margin.csv",
    index=False,
)

### Inhomogeneous-$D_r$ thermal margin

This path keeps a translation-breaking, phase-compatible bond background fixed and changes $D_r$ along a unit Euclidean direction.  Exactness is guaranteed by the analytical diagonal condition; the finite-temperature directed activity and its susceptibility are measured numerically.

In [ ]:
L_MARGIN_INHOM = 6
rng_margin = np.random.default_rng(23)
inhom_margin_pairs = []
for distance, offset, width in ((1, 1.2, 0.7), (3, 0.2, 0.6)):
    for site_i, site_j, _ in spin_one_xy_periodic_range_couplings(
        length=L_MARGIN_INHOM,
        distance=distance,
        coefficient=1.0,
    ):
        inhom_margin_pairs.append(
            (site_i, site_j, float(offset + width * rng_margin.random()))
        )
base_d_profile = 0.35 + 0.45 * rng_margin.random(L_MARGIN_INHOM)
d_direction = rng_margin.normal(size=L_MARGIN_INHOM)
d_direction /= np.linalg.norm(d_direction)
G_PATH = np.linspace(-0.20, 0.20, 5)
inhom_margin_rows = []
inhom_base_conditioning = None
for g_value in G_PATH:
    profile = base_d_profile + float(g_value) * d_direction
    model = SpinOneXYChainModel(
        length=L_MARGIN_INHOM,
        boundary_condition="periodic",
        j_xy=0.0,
        d_z_by_site=tuple(float(value) for value in profile),
        total_sz=TOTAL_SZ,
        extra_xy_couplings=tuple(inhom_margin_pairs),
    )
    result = model.build(builder="optimized", basis_solver="dfs", sort_basis=True)
    configs = basis_configs_from_build_result(result)
    scar = tower_state_for_sector(configs, length=L_MARGIN_INHOM)
    energies, vectors = la.eigh(result.hamiltonian.toarray())
    q_local = A_UNIT.embed(configs)
    activities = eigenstate_expectations(q_local.conj().T @ q_local, vectors)
    scar_energy = float(np.sum(profile))
    window = select_microcanonical_window_by_count(
        energies,
        target_energy=scar_energy,
        target_count=min(24, energies.size),
        include_boundary_degeneracy=True,
    )
    indices = np.asarray(window.indices, dtype=np.int64)
    if abs(float(g_value)) <= TOL:
        inhom_base_support = np.flatnonzero(np.abs(scar) > TOL)
        inhom_base_conditioning = cage_jacobian_conditioning_from_hamiltonian(
            result.hamiltonian,
            inhom_base_support,
            scar,
            tolerance=TOL,
        )
    inhom_margin_rows.append(
        {
            "g": float(g_value),
            "scar_energy": scar_energy,
            "scar_residual": diagnose_eigenpair(result.hamiltonian, scar).residual_norm,
            "window_state_count": window.n_states,
            "window_half_width": window.half_width,
            "unit_directed_activity": float(np.mean(activities[indices])),
        }
    )
if inhom_base_conditioning is None:
    raise RuntimeError("the inhomogeneous path must include g=0")
inhom_margin_df = pd.DataFrame(inhom_margin_rows)
inhom_margin = thermal_activity_margin_from_samples(
    inhom_margin_df["g"],
    inhom_margin_df["unit_directed_activity"],
    reference_parameter=0.0,
    tolerance=TOL,
)
display(inhom_margin_df)
display(pd.DataFrame([inhom_margin.to_summary_dict()]))
inhom_margin_df.to_csv(DATA_DIR / "inhomogeneous_D_directed_thermal_path.csv", index=False)
pd.DataFrame([inhom_margin.to_summary_dict()]).to_csv(
    DATA_DIR / "inhomogeneous_D_directed_thermal_margin.csv",
    index=False,
)

stability_profile_df = pd.DataFrame(
    [
        {
            "case": "uniform_finite_D",
            "L": L_STABILITY,
            "Delta_cage": cage_conditioning.cage_gap,
            "witness_radius": 1,
            "Delta_Q": local_q_spectrum.dark_channel_gap,
            "tau_Q": finite_d_margin.reference_activity,
            "chi_Q": finite_d_margin.susceptibility_bound,
            "half_activity_radius": finite_d_margin.half_activity_radius,
        },
        {
            "case": "inhomogeneous_Dr",
            "L": L_MARGIN_INHOM,
            "Delta_cage": inhom_base_conditioning.cage_gap,
            "witness_radius": 1,
            "Delta_Q": local_q_spectrum.dark_channel_gap,
            "tau_Q": inhom_margin.reference_activity,
            "chi_Q": inhom_margin.susceptibility_bound,
            "half_activity_radius": inhom_margin.half_activity_radius,
        },
    ]
)
display(stability_profile_df)
stability_profile_df.to_csv(DATA_DIR / "predictive_stability_profile.csv", index=False)

### Draft-oriented thermal-margin figures
These plots translate the predictive deformation quantities into presentation-ready curves: the directed-witness thermal activity along a compatible uniform-$D$ path, the analogous path in the inhomogeneous-$D_r$ family, and a compact summary of $(\Delta_{\rm cage},\Delta_Q,\tau_Q,\chi_Q)$ for the two benchmark cases.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(finite_d_margin_df['coupling_path_parameter'], finite_d_margin_df['unit_directed_activity'], marker='o')
ax.axhline(finite_d_margin.reference_activity, linestyle='--', label=r'$\tau_Q$ at $g=0$')
ax.set_xlabel(r'uniform-$D$ path parameter $g$')
ax.set_ylabel(r'microcanonical $\langle A^\dagger A\rangle$')
ax.legend(loc='best', fontsize=9)
ax.grid()
fig.tight_layout()
fig.savefig(DATA_DIR / 'finite_D_directed_thermal_margin_curve.pdf')

fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(inhom_margin_df['g'], inhom_margin_df['unit_directed_activity'], marker='o')
ax.axhline(inhom_margin.reference_activity, linestyle='--', label=r'$\tau_Q$ at $g=0$')
ax.set_xlabel(r'inhomogeneous-$D_r$ path parameter $g$')
ax.set_ylabel(r'microcanonical $\langle A^\dagger A\rangle$')
ax.legend(loc='best', fontsize=9)
ax.grid()
fig.tight_layout()
fig.savefig(DATA_DIR / 'inhomogeneous_D_directed_thermal_margin_curve.pdf')

summary_plot_df = stability_profile_df.copy()
summary_plot_df = summary_plot_df.set_index('case')
for quantity, filename in [
    ('Delta_cage', 'predictive_stability_delta_cage.pdf'),
    ('Delta_Q', 'predictive_stability_delta_Q.pdf'),
    ('tau_Q', 'predictive_stability_tau_Q.pdf'),
    ('chi_Q', 'predictive_stability_chi_Q.pdf'),
]:
    fig, ax = plt.subplots(figsize=(5.8, 3.6))
    ax.bar(summary_plot_df.index.tolist(), summary_plot_df[quantity].to_numpy())
    ax.set_ylabel(quantity)
    ax.set_xlabel('benchmark case')
    ax.grid(axis='y')
    fig.tight_layout()
    fig.savefig(DATA_DIR / filename)


## G. Data manifest

In [ ]:
manifest = pd.DataFrame(
    [
        {"file": path.name, "bytes": path.stat().st_size}
        for path in sorted(DATA_DIR.glob("*.csv"))
    ]
)
display(manifest)
manifest.to_csv(DATA_DIR / "manifest.csv", index=False)
print("All numerical tables were written to", DATA_DIR)

## Interpretation checklist for the manuscript

- **Analytical:** tower wavefunction, boundary kernel, all three local annihilators, exact $D=0$ fixed-$M$ activities, and the preserving bond/diagonal conditions.
- **Numerically corroborated:** residuals, direct traces, and the local/full obstruction ranks at modest size.
- **Thermal-background evidence:** symmetry-resolved ETH scatter and level statistics at finite $D$.
- **Directed witness:** $Q^{\rm in}=A^\dagger A$ is one-sided and weaker than $Z^2$; it need not test local occupation of $|00\rangle$ in complementary sectors.
- **Predictive stability:** obstruction rank tests compatibility; $\Delta_{\rm cage}$ and $\Delta_Q$ test conditioning only after compatibility; $(\tau_Q,\chi_Q)$ give a finite-size thermal margin.
- **Finite $D$ and inhomogeneous $D_r$:** the scar remains exact, but the thermal activity is no longer given by the simple $D=0$ counting and is measured directly.
- **Caution:** every margin and gap in this notebook is finite-size.  A thermodynamic robustness claim requires size-uniform lower bounds and bounded deformation derivatives.